# Populate Northstar data sources

Run this notebook as an instructor before the workshop. The write cell overwrites the eight Northstar tables in the selected catalog and schema. Use a dedicated workshop schema, not a schema containing unrelated data.

After writing the tables, synchronize the managed AI Search index on `document_chunks` and verify retrieval before class. This notebook does not create or synchronize the index.

In [ ]:
from pathlib import Path
import sys

repository_root = Path.cwd()
if not (repository_root / 'instructor' / 'scripts').is_dir():
    repository_root = repository_root.parent
assert (repository_root / 'instructor' / 'scripts').is_dir(), 'Run from the repository or instructor folder.'
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from instructor.scripts.northstar_data import build_northstar_dataframes, write_northstar_tables

dbutils.widgets.text('catalog', '', 'Unity Catalog')
dbutils.widgets.text('schema', '', 'Dedicated workshop schema')
dbutils.widgets.text('overwrite_confirmation', '', 'Type OVERWRITE to write tables')

CATALOG = dbutils.widgets.get('catalog').strip()
SCHEMA = dbutils.widgets.get('schema').strip()
OVERWRITE_CONFIRMATION = dbutils.widgets.get('overwrite_confirmation').strip()
assert CATALOG and SCHEMA, 'Enter the catalog and dedicated schema.'
assert SCHEMA.lower() != 'default', 'Choose a dedicated workshop schema.'
assert OVERWRITE_CONFIRMATION == 'OVERWRITE', 'Confirm the table overwrite before continuing.'

## Preview

Build the deterministic DataFrames and inspect their row counts before writing. This cell does not modify Unity Catalog.

In [ ]:
frames = build_northstar_dataframes(spark)
for table_name, dataframe in frames.items():
    print(f'{table_name}: {dataframe.count():,} rows')

display(frames['sales_weekly'].filter(
    (frames['sales_weekly'].product_id == 'OAT-1')
    & (frames['sales_weekly'].region == 'Midwest')
).limit(10))

## Write tables

Run only after checking the catalog, schema, preview and overwrite confirmation. The generator replaces existing tables with these names.

In [ ]:
write_report = write_northstar_tables(spark, CATALOG, SCHEMA)
for table_name, details in write_report.items():
    print(f"{table_name}: {details['table']} ({details['rows']:,} rows)")

display(spark.table(write_report['sales_weekly']['table']).filter(
    (spark.table(write_report['sales_weekly']['table']).product_id == 'OAT-1')
    & (spark.table(write_report['sales_weekly']['table']).region == 'Midwest')
).limit(10))